# Fine-tune STEMI — khởi tạo từ backbone pretrain (notebook 14)

**Dự án ACS-ECG-AI (Vinmec).** Notebook 15 — fine-tune nhánh **STEMI** trên ACS-ECG 2026,
khởi tạo backbone từ checkpoint pretrain (PTB-XL supervised + MIMIC-IV-ECG self-supervised) thay vì
random init, theo đúng thiết kế đã bàn (Phương án A).

**Trạng thái: đã nối đầy đủ với dữ liệu thật, chỉ cần notebook 14 chạy xong (`RUN_MODE="full"`)
rồi đổi `RUN_MODE = "full"` ở đây và chạy toàn bộ notebook.** Bước 2 đọc thẳng cùng nguồn dữ liệu
và lặp lại đúng cách chia POOL/TEST + K-Fold của `stemi_pipeline_optimized_v3.ipynb` (cùng `SEED`)
— nên TEST và fold assignment ở đây khớp chính xác với baseline from-scratch.

**Quy trình (khớp nguyên tắc GPU-cost-tiered đã dùng cho ECGFounder ở Phần I):**
1. Load backbone checkpoint + manifest từ notebook 14, xác nhận config khớp (FS, SIGNAL_LEN, preprocess).
2. **Screening 1-fold** qua 6 mức freeze/unfreeze (rẻ) → chọn mức thắng theo AUPRC.
3. **Full K-fold** (5-fold) chỉ với mức freeze/unfreeze thắng → sinh OOF cho toàn POOL.
4. Khoá threshold trên POOL-OOF (Sensitivity ≥91% làm sàn, tối đa hoá Specificity/NPV).
5. Train FINAL trên toàn POOL, đánh giá TEST giữ riêng, so sánh bootstrap ΔAUPRC với baseline
   from-scratch hiện có trong `Bao_cao_nghien_cuu.docx`.

**Còn lại cần tay (không chặn chạy `RUN_MODE="full"`):** Bước 10 — `baseline_y_prob` để so sánh
bootstrap. Champion của baseline luôn là một ensemble (trọng số fit lúc chạy, không lưu ra file),
nên không thể tự động tái tạo chính xác — cần điền thủ công xác suất TEST của baseline (đọc từ
`stemi_test_predictions_stemi_optimized_v3.npz`, xem hướng dẫn ở Bước 10).

**Nguyên tắc bắt buộc:**
- Threshold khoá trên POOL-OOF, áp KHÔNG ĐỔI lên TEST — không bao giờ refit trên TEST.
- Patient-level split, K-Fold **giống hệt** `stemi_pipeline_optimized_v3.ipynb` (cùng SEED) để so
  sánh công bằng với baseline from-scratch.
- TEST chỉ được chạm ở Bước 9 (đánh giá cuối cùng), không dùng cho bất kỳ lựa chọn nào trước đó.

In [2]:
import os, sys, json, hashlib, random, time, re, shutil
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd

try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    from torch.utils.data import Dataset, DataLoader
except ImportError:
    raise SystemExit("Chưa có torch — chạy trên Colab GPU runtime (Runtime > Change runtime type > GPU).")

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import roc_curve, confusion_matrix, average_precision_score, roc_auc_score

from google.colab import drive
drive.mount('/content/drive')

# ============================== CONFIG ==============================
RUN_MODE = "full"  # "debug" | "full"

SEED = 42
FS = 500
SIGNAL_LEN = 5000
NUM_LEADS = 12
N_FOLDS = 5
DEBUG_LIMIT = 200  # số record dùng khi RUN_MODE=debug -- pipeline chạy THẬT, không giả lập

TASK = "stemi"                 # "stemi" | "omi"
TARGET_COL = "STEMI"                 # cột nhãn thật trong CSV/train.csv -- khớp stemi_pipeline_optimized_v3.ipynb mục 5
PATIENT_COL = "Patient_id"           # cột bệnh nhân thật -- khớp COL_PATIENT của stemi_pipeline_optimized_v3.ipynb
EXISTING_FOLD_COL = "fold"           # gán trong load_pool_and_test() (Bước 2) -- tái lập ĐÚNG fold
                                       # StratifiedKFold patient-level của stemi_pipeline_optimized_v3.ipynb
                                       # mục 8 (cùng SEED) để so sánh công bằng với baseline from-scratch

THRESHOLD_POLICY = "sensitivity_floor"   # "sensitivity_floor" | "youden"
SENSITIVITY_FLOOR = 0.91                   # chỉ dùng khi THRESHOLD_POLICY="sensitivity_floor"

RUN_TAG_SUFFIX = "_tier1"   # đổi tên checkpoint so với Tier 0 -- BẮT BUỘC train lại từ đầu với
                              # Focal Loss + augmentation + warmup/schedule mới, không âm thầm
                              # resume/dùng lại checkpoint Tier 0 (huấn luyện dưới cấu hình khác).

# --- Tier 1: augmentation khi train (verbatim stemi_pipeline_optimized_v3.ipynb mục 9) ---
# Áp thẳng trên tín hiệu đã tiền xử lý (mV, KHÔNG z-score) -- backbone pretrain ở notebook 14
# cũng không z-score, giữ nguyên phân phối đầu vào mà backbone đã học.
AUG_ENABLE = True
AUG_MAX_SHIFT = 20            # dịch thời gian tối đa (mẫu, ~40ms @ 500Hz)
AUG_NOISE_STD = 0.02          # nhiễu Gaussian biên độ thấp (mV)
AUG_GAIN_JITTER = 0.05        # hệ số gain ngẫu nhiên ±5%/chuyển đạo
AUG_OP_PROB = 0.5             # xác suất áp mỗi phép augment gốc (3 phép đầu)
AUG_POWERLINE_PROB = 0.3      # xác suất chồng nhiễu điện lưới
AUG_POWERLINE_HZ = 50.0
AUG_POWERLINE_STD = 0.03      # biên độ nhiễu điện lưới (mV)
AUG_LEAD_DROPOUT_PROB = 0.03  # xác suất một chuyển đạo bị thay bằng nhiễu thấp

# --- Tier 1: Focal Loss + label smoothing (verbatim mục 12) -- bù mất cân bằng lớp dương ---
FOCAL_GAMMA = 2.0
LABEL_SMOOTH_EPS = 0.02

# --- Tier 1: warmup + ReduceLROnPlateau + early stopping (verbatim mục 2/13) ---
WARMUP_EPOCHS = 3
LR_PATIENCE = 5
EARLY_STOP_PATIENCE = 10

# --- Nguồn dữ liệu ACS-ECG 2026 thật (KHÁC Drive project của backbone pretrain ở dưới) ---
# Cùng chuẩn với stemi_pipeline_optimized_v3.ipynb mục 3-5: train.csv trong CSV/, waveform
# .dat/.hea trong row_data/ (hoặc raw_data/), cùng danh sách bản ghi lỗi đọc thật ở nguồn.
DATA_DRIVE_PROJECT = Path("/content/drive/MyDrive/ACS-ECG-AI")
DATA_DRIVE_ZIP = DATA_DRIVE_PROJECT / "datasets.zip"
DATA_ROOT = Path("/content/acs_ecg_datasets")
KNOWN_BROKEN_RECORDS = ["03228", "14262"]   # verbatim từ stemi_pipeline_optimized_v3.ipynb mục 2
TEST_SIZE = 0.15                            # verbatim mục 2 -- 15% bệnh nhân giữ riêng làm TEST

DRIVE_ROOT = Path("/content/drive/MyDrive/ACS-ECG-AI_pretrain_finetune")
PRETRAIN_MANIFEST_DIR = DRIVE_ROOT / "manifests"
PRETRAIN_MODELS_DIR = DRIVE_ROOT / "models" / "backbone_checkpoints"

FINETUNE_MODELS_DIR = DRIVE_ROOT / "models" / "stemi_finetune"
FINETUNE_LOGS_DIR = DRIVE_ROOT / "logs" / "stemi_finetune"
FINETUNE_OOF_DIR = DRIVE_ROOT / "manifests" / "stemi_finetune_oof"
LOCAL_SIGNAL_CACHE_DIR = Path("/content/stemi_finetune_work/cache")
DRIVE_SIGNAL_CACHE_DIR = DRIVE_ROOT / "data" / "stemi_signal_cache"
for d in [FINETUNE_MODELS_DIR, FINETUNE_LOGS_DIR, FINETUNE_OOF_DIR,
          LOCAL_SIGNAL_CACHE_DIR, DRIVE_SIGNAL_CACHE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"TASK={TASK} | RUN_MODE={RUN_MODE} | device={DEVICE}")

Mounted at /content/drive
TASK=stemi | RUN_MODE=full | device=cuda


## Bước 1 — Load backbone checkpoint từ notebook 14, xác nhận config khớp

Không chỉ load trọng số mù quáng — kiểm tra `run_manifest.json` để chắc `FS`, `SIGNAL_LEN`,
`NUM_LEADS`, thông số filter/winsorize đã dùng lúc pretrain khớp với pipeline fine-tune ở đây,
tránh lỗi âm thầm (ví dụ pretrain dùng highpass khác, tiền xử lý lệch nhau).

In [3]:
def find_latest_backbone_manifest(manifest_dir=PRETRAIN_MANIFEST_DIR):
    candidates = sorted(manifest_dir.glob("*_run_manifest.json"), key=lambda p: p.stat().st_mtime)
    if not candidates:
        raise FileNotFoundError(
            f"Không tìm thấy *_run_manifest.json trong {manifest_dir} -- chạy notebook 14 "
            f"(RUN_MODE=\"full\") trước khi chạy notebook này."
        )
    return candidates[-1]

_manifest_path = find_latest_backbone_manifest()
_manifest = json.loads(_manifest_path.read_text())
print(f"Dùng backbone: {_manifest_path.name}")
print(json.dumps(_manifest, indent=2, ensure_ascii=False))

_pretrain_cfg = _manifest["config"]
assert _pretrain_cfg["FS"] == FS, f"FS lệch: pretrain={_pretrain_cfg['FS']} vs finetune={FS}"
assert _pretrain_cfg["SIGNAL_LEN"] == SIGNAL_LEN, "SIGNAL_LEN lệch giữa pretrain và finetune"
assert _pretrain_cfg["NUM_LEADS"] == NUM_LEADS, "NUM_LEADS lệch giữa pretrain và finetune"
BP_LOW = _pretrain_cfg.get("BP_LOW", 0.05)
BP_HIGH = _pretrain_cfg.get("BP_HIGH", 40.0)
BP_ORDER = _pretrain_cfg.get("BP_ORDER", 3)
WINSORIZE_MV = _pretrain_cfg.get("WINSORIZE_MV", 6.0)
print(f"\nTiền xử lý kế thừa từ pretrain: BP=[{BP_LOW},{BP_HIGH}]Hz order={BP_ORDER}, "
      f"winsorize=±{WINSORIZE_MV}mV")

BACKBONE_CKPT_PATH = Path(_manifest["backbone_checkpoint"])
_expected_sha256 = _manifest["sha256"]
_actual_sha256 = hashlib.sha256(BACKBONE_CKPT_PATH.read_bytes()).hexdigest()
assert _actual_sha256 == _expected_sha256, (
    "Checksum backbone checkpoint không khớp manifest -- file có thể đã bị sửa/hỏng, chạy lại "
    "notebook 14 hoặc kiểm tra lại đường dẫn."
)
print(f"\nBackbone checkpoint: {BACKBONE_CKPT_PATH} (SHA-256 khớp manifest)")

Dùng backbone: resnet1d_real_pretrain_run_manifest.json
{
  "run_tag": "resnet1d_real_pretrain",
  "created_at": "2026-08-27T15:33:45.416247",
  "run_mode": "full",
  "source_best_checkpoint": "/content/drive/MyDrive/ACS-ECG-AI_pretrain_finetune/models/backbone_checkpoints/resnet1d_real_pretrain_best.pt",
  "backbone_checkpoint": "/content/drive/MyDrive/ACS-ECG-AI_pretrain_finetune/models/backbone_checkpoints/resnet1d_real_pretrain_backbone_only.pt",
  "sha256": "df0789dc15e020b34090f9bf84d054369e1fe8ffddd52d4029a8cd1256154efa",
  "backbone_architecture": "ResNet1D (verbatim tu stemi_pipeline_optimized_v3.ipynb muc 11)",
  "pretrain_design": "PTB-XL supervised + MIMIC-IV-ECG self-supervised (NT-Xent), khong can credential",
  "n_ptbxl": 21799,
  "n_mimic": 89970,
  "config": {
    "SEED": 42,
    "FS": 500,
    "SIGNAL_LEN": 5000,
    "NUM_LEADS": 12,
    "BP_LOW": 0.05,
    "BP_HIGH": 40.0,
    "BP_ORDER": 3,
    "WINSORIZE_MV": 6.0,
    "PREPROCESS_VERSION": "pretrain_v3_real",
    "

## Bước 2 — Load POOL/TEST thật của ACS-ECG 2026

`load_pool_and_test()` đọc thẳng cùng nguồn dữ liệu và lặp lại **đúng** logic chia
POOL/TEST + K-Fold của `stemi_pipeline_optimized_v3.ipynb` (mục 3, 5, 7, 8), cùng `SEED` —
nên tập TEST và fold assignment ở đây **khớp chính xác** với baseline from-scratch, cho phép so
sánh bootstrap công bằng ở Bước 10:

- Giải nén `datasets.zip` từ `ACS-ECG-AI/` trên Drive (project dữ liệu gốc — khác
  `ACS-ECG-AI_pretrain_finetune/` chứa backbone pretrain).
- Đọc `CSV/train.csv`, nhãn lấy trực tiếp từ cột `STEMI`, loại 2 bản ghi lỗi đọc thật ở nguồn
  (`KNOWN_BROKEN_RECORDS`, verbatim từ pipeline gốc).
- Tiền xử lý (bandpass + winsorize) dùng đúng `BP_LOW/BP_HIGH/BP_ORDER/WINSORIZE_MV` đã xác nhận
  khớp với pretrain ở Bước 1, cache ra 1 file `.npy` memmap resumable (cùng nguyên tắc notebook 14
  Bước 4) — POOL và TEST chỉ là 2 view khác nhau trên cùng một cache, không tách vật lý.
- Chia POOL/TEST theo bệnh nhân (`train_test_split`, `test_size=TEST_SIZE`, `random_state=SEED+200`)
  và K-Fold patient-level (`StratifiedKFold`, `random_state=SEED`) — **cùng tham số** mục 7-8 của
  pipeline gốc nên tái lập đúng tập bệnh nhân TEST và fold assignment.

In [4]:
import importlib.util
import subprocess
import zipfile

if importlib.util.find_spec("wfdb") is None:
    print("Đang cài wfdb ...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "wfdb"], check=True)
import wfdb

from scipy.signal import butter, filtfilt
from sklearn.model_selection import StratifiedKFold, train_test_split


def find_dir(root: Path, names):
    wanted = {n.lower() for n in names}
    for c in sorted(root.iterdir()):
        if c.is_dir() and c.name.lower() in wanted:
            return c
    for c in root.rglob("*"):
        if c.is_dir() and c.name.lower() in wanted:
            return c
    return None


def stage_acs_ecg_data(data_root=DATA_ROOT, drive_zip=DATA_DRIVE_ZIP):
    """Giải nén datasets.zip của ACS-ECG-AI về đĩa local Colab -- CÙNG file zip mà
    stemi_pipeline_optimized_v3.ipynb dùng (mục 3), khác Drive project riêng của backbone
    pretrain (notebook 14 dùng ACS-ECG-AI_pretrain_finetune/)."""
    marker = data_root / ".staged_ok"
    if marker.exists():
        real = Path(marker.read_text().strip())
        print(f"Dữ liệu ACS-ECG đã sẵn sàng: {real}")
        return real
    if not drive_zip.exists():
        raise FileNotFoundError(
            f"Không thấy {drive_zip} -- kiểm tra thư mục ACS-ECG-AI trong MyDrive "
            f"(cùng thư mục stemi_pipeline_optimized_v3.ipynb đang dùng)."
        )
    local_zip = Path("/content/_acs_ecg_dataset.zip")
    size = drive_zip.stat().st_size
    if not (local_zip.exists() and local_zip.stat().st_size == size):
        print(f"Copy zip {size / 1024 ** 3:.2f} GB từ Drive ...")
        t0 = time.time()
        shutil.copy2(drive_zip, local_zip)
        print(f"  {time.time() - t0:.0f}s")
    extract_to = Path("/content/_acs_ecg_extract")
    if extract_to.exists():
        shutil.rmtree(extract_to)
    print("Giải nén ...")
    t0 = time.time()
    with zipfile.ZipFile(local_zip) as zf:
        zf.extractall(extract_to)
    print(f"  {time.time() - t0:.0f}s")

    real = extract_to
    if find_dir(extract_to, ["csv"]) is None:
        for cand in sorted(p for p in extract_to.rglob("*") if p.is_dir()):
            if find_dir(cand, ["csv"]) is not None:
                real = cand
                break
    data_root.mkdir(parents=True, exist_ok=True)
    marker.write_text(str(real))
    return real


def load_signal(stem: str, raw_dir: Path) -> np.ndarray:
    """Verbatim load_signal() của stemi_pipeline_optimized_v3.ipynb mục 6 (đọc đúng số mẫu
    thực có khi .dat ngắn hơn header khai báo, thay vì để wfdb raise ValueError)."""
    path = str(raw_dir / stem)
    try:
        rec = wfdb.rdrecord(path)
    except ValueError:
        n_sig = int((raw_dir / f"{stem}.hea").read_text().splitlines()[0].split()[1])
        n = (raw_dir / f"{stem}.dat").stat().st_size // (n_sig * 2)
        rec = wfdb.rdrecord(path, sampto=n)
    return np.asarray(rec.p_signal, dtype=np.float32).T


_B, _A = butter(BP_ORDER, [BP_LOW / (FS / 2), BP_HIGH / (FS / 2)], btype="band")


def preprocess_signal(sig: np.ndarray) -> np.ndarray:
    """Verbatim preprocess() của stemi_pipeline_optimized_v3.ipynb mục 6 -- BP_LOW/BP_HIGH/
    BP_ORDER/WINSORIZE_MV kế thừa từ manifest pretrain ở Bước 1 (đã xác nhận khớp)."""
    sig = np.nan_to_num(sig, nan=0.0, posinf=0.0, neginf=0.0)
    if sig.shape[1] < SIGNAL_LEN:
        sig = np.pad(sig, ((0, 0), (0, SIGNAL_LEN - sig.shape[1])))
    filtered = filtfilt(_B, _A, sig[:, :SIGNAL_LEN], axis=1)
    winsorized = np.clip(filtered, -WINSORIZE_MV, WINSORIZE_MV)
    return np.ascontiguousarray(winsorized, dtype=np.float32)


def _cache_complete(meta_p, npy_p, n_expected):
    if not (meta_p.exists() and npy_p.exists()):
        return False
    try:
        return json.loads(meta_p.read_text()).get("n_done", 0) >= n_expected
    except (OSError, ValueError):
        return False


def build_signal_cache(record_ids, raw_dir, tag, checkpoint_every=2000):
    """Cache resumable 1 file .npy lớn (memmap), build local trước rồi backup định kỳ + cuối
    cùng lên Drive -- cùng nguyên tắc notebook 14 Bước 4 / pipeline gốc mục 6."""
    n = len(record_ids)
    npy_p = LOCAL_SIGNAL_CACHE_DIR / f"{tag}.npy"
    meta_p = LOCAL_SIGNAL_CACHE_DIR / f"{tag}.meta.json"
    d_npy_p = DRIVE_SIGNAL_CACHE_DIR / f"{tag}.npy"
    d_meta_p = DRIVE_SIGNAL_CACHE_DIR / f"{tag}.meta.json"

    if not _cache_complete(meta_p, npy_p, n) and _cache_complete(d_meta_p, d_npy_p, n):
        print(f"[{tag}] Cache đầy đủ trên Drive -- copy về local ...")
        shutil.copy2(d_npy_p, npy_p)
        shutil.copy2(d_meta_p, meta_p)

    if _cache_complete(meta_p, npy_p, n):
        print(f"[{tag}] Cache đã đầy đủ ({n:,} bản ghi), dùng lại.")
        return np.load(npy_p, mmap_mode="r")

    meta = json.loads(meta_p.read_text()) if meta_p.exists() else {}
    start = int(meta.get("n_done", 0)) if npy_p.exists() else 0
    if start:
        arr = np.lib.format.open_memmap(npy_p, mode="r+")
        print(f"[{tag}] Build tiếp từ {start:,}/{n:,}")
    else:
        arr = np.lib.format.open_memmap(npy_p, mode="w+", dtype=np.float16,
                                        shape=(n, NUM_LEADS, SIGNAL_LEN))
        print(f"[{tag}] Build cache mới {n:,} bản ghi "
              f"(~{n * NUM_LEADS * SIGNAL_LEN * 2 / 1024 ** 3:.2f} GB)")

    t0 = time.time()
    for i in range(start, n):
        arr[i] = preprocess_signal(load_signal(record_ids[i], raw_dir)).astype(np.float16)
        done = i + 1
        if done % 500 == 0 or done == n:
            el = max(time.time() - t0, 1e-6)
            print(f"  [{tag}] {done:>7,}/{n:,}  {(done - start) / el:5.1f} rec/s")
        if done % checkpoint_every == 0 or done == n:
            arr.flush()
            meta_p.write_text(json.dumps({"n_done": done}))
            shutil.copy2(npy_p, d_npy_p)
            shutil.copy2(meta_p, d_meta_p)
    del arr
    print(f"[{tag}] Xong trong {time.time() - t0:.0f}s, đã lưu Drive: {d_npy_p}")
    return np.load(npy_p, mmap_mode="r")


def load_pool_and_test():
    """Đọc thật POOL/TEST của ACS-ECG 2026 -- CÙNG nguồn dữ liệu, CÙNG cách chia bệnh nhân/fold
    (cùng SEED) với stemi_pipeline_optimized_v3.ipynb (mục 3-8), để TEST và fold assignment
    KHỚP CHÍNH XÁC với baseline from-scratch dùng cho so sánh bootstrap ở Bước 10.

    Trả về: pool_df, test_df (cột record_id/{PATIENT_COL}/{TARGET_COL}/_cache_idx, pool_df có
    thêm cột {EXISTING_FOLD_COL}), pool_cache, test_cache -- CÙNG một mảng memmap, index theo
    _cache_idx (POOL/TEST chỉ là 2 view khác nhau trên cùng cache, không tách vật lý).
    """
    data_root = stage_acs_ecg_data()
    raw_dir = find_dir(data_root, ["row_data", "raw_data"])
    csv_dir = find_dir(data_root, ["csv"])
    assert raw_dir and csv_dir, f"Không thấy row_data/raw_data hoặc CSV trong {data_root}"

    df_raw = pd.read_csv(csv_dir / "train.csv")
    df_raw["record_id"] = df_raw["ecg_row_record"].astype(str).str.replace(".dat", "", regex=False)
    df_raw[TARGET_COL] = df_raw[TARGET_COL].astype(int)

    _broken = df_raw["record_id"].isin(KNOWN_BROKEN_RECORDS)
    if _broken.any():
        print(f"Loại {int(_broken.sum())} bản ghi lỗi đọc thật ở nguồn: "
              f"{df_raw.loc[_broken, 'record_id'].tolist()}")
        df_raw = df_raw[~_broken].reset_index(drop=True)

    df_all = df_raw[["record_id", PATIENT_COL, TARGET_COL]].copy()

    if RUN_MODE == "debug" and len(df_all) > DEBUG_LIMIT:
        df_all, _ = train_test_split(df_all, train_size=DEBUG_LIMIT,
                                     stratify=df_all[TARGET_COL], random_state=SEED)
    df_all = df_all.reset_index(drop=True)

    n_pos = int(df_all[TARGET_COL].sum())
    print(f"{TASK.upper()}: {len(df_all):,} bản ghi | {n_pos:,} dương ({n_pos / len(df_all):.2%})")
    assert 0 < n_pos < len(df_all), "Tập chỉ có một lớp -- không train được."

    preprocess_tag = f"hp{BP_LOW}_{BP_HIGH}hz_o{BP_ORDER}_ws{WINSORIZE_MV}"
    full_cache = build_signal_cache(df_all["record_id"].tolist(), raw_dir,
                                    tag=f"{TASK}_{preprocess_tag}_n{len(df_all)}")
    df_all["_cache_idx"] = np.arange(len(df_all))

    # --- Chia POOL/TEST theo bệnh nhân -- ĐÚNG stemi_pipeline_optimized_v3.ipynb mục 7,
    # cùng SEED+200 -> tái lập CHÍNH XÁC cùng bệnh nhân TEST của baseline from-scratch ---
    pat_all = df_all.groupby(PATIENT_COL)[TARGET_COL].max().reset_index()
    pat_pool, pat_test = train_test_split(pat_all, test_size=TEST_SIZE,
                                          stratify=pat_all[TARGET_COL], random_state=SEED + 200)
    test_patients = set(pat_test[PATIENT_COL])
    is_test = df_all[PATIENT_COL].isin(test_patients).to_numpy()
    pool_df = df_all[~is_test].reset_index(drop=True)
    test_df = df_all[is_test].reset_index(drop=True)
    assert not (set(pool_df[PATIENT_COL]) & test_patients), "Rò rỉ bệnh nhân TEST vào POOL"

    # --- K-fold patient-level trong POOL -- ĐÚNG mục 8, cùng SEED -> tái lập CHÍNH XÁC cùng
    # fold assignment của baseline (miễn POOL patient set giống nhau, đã đảm bảo ở bước trên) ---
    pat_tbl = pool_df.groupby(PATIENT_COL)[TARGET_COL].max().reset_index()
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    fold_of_patient = {}
    for k, (_, va_pos) in enumerate(skf.split(pat_tbl, pat_tbl[TARGET_COL])):
        for pid in pat_tbl.iloc[va_pos][PATIENT_COL]:
            fold_of_patient[pid] = k
    pool_df[EXISTING_FOLD_COL] = pool_df[PATIENT_COL].map(fold_of_patient)
    assert pool_df[EXISTING_FOLD_COL].notna().all(), "Có bệnh nhân POOL chưa được gán fold"
    pool_df[EXISTING_FOLD_COL] = pool_df[EXISTING_FOLD_COL].astype(int)

    print(f"POOL: {len(pool_df):,} bản ghi / {pool_df[PATIENT_COL].nunique():,} bệnh nhân | "
          f"TEST: {len(test_df):,} bản ghi / {test_df[PATIENT_COL].nunique():,} bệnh nhân")
    print(f"Tỷ lệ dương POOL: {pool_df[TARGET_COL].mean():.2%} | TEST: {test_df[TARGET_COL].mean():.2%}")
    return pool_df, test_df, full_cache, full_cache


pool_df, test_df, pool_cache, test_cache = load_pool_and_test()

Đang cài wfdb ...
Copy zip 1.33 GB từ Drive ...
  22s
Giải nén ...
  29s
Loại 2 bản ghi lỗi đọc thật ở nguồn: ['14262', '03228']
STEMI: 17,958 bản ghi | 1,442 dương (8.03%)
[stemi_hp0.05_40.0hz_o3_ws6.0_n17958] Cache đầy đủ trên Drive -- copy về local ...
[stemi_hp0.05_40.0hz_o3_ws6.0_n17958] Cache đã đầy đủ (17,958 bản ghi), dùng lại.
POOL: 15,281 bản ghi / 14,463 bệnh nhân | TEST: 2,677 bản ghi / 2,553 bệnh nhân
Tỷ lệ dương POOL: 8.02% | TEST: 8.11%


## Bước 3 — K-fold patient-level (tái dùng fold có sẵn nếu POOL đã gán từ trước)

In [5]:
def make_or_reuse_folds(pool_df, n_folds=N_FOLDS, seed=SEED,
                         patient_col=None, existing_fold_col=EXISTING_FOLD_COL, target_col=None):
    patient_col = patient_col or PATIENT_COL
    target_col = target_col or TARGET_COL
    if existing_fold_col in pool_df.columns:
        print(f"Dùng lại cột fold có sẵn '{existing_fold_col}' -- khớp fold của baseline from-scratch.")
        return pool_df[existing_fold_col].to_numpy()
    print(f"Không thấy cột '{existing_fold_col}' -- tự chia {n_folds}-fold patient-level mới "
          f"(StratifiedGroupKFold). LƯU Ý: fold này sẽ KHÁC baseline nếu baseline dùng fold khác --"
          f" so sánh bootstrap ở Bước 7 vẫn hợp lệ (cùng TEST) nhưng OOF không trực tiếp so được.")
    sgkf = StratifiedGroupKFold(n_splits=n_folds, shuffle=True, random_state=seed)
    fold_assign = np.full(len(pool_df), -1, dtype=int)
    for fold_id, (_, val_idx) in enumerate(
            sgkf.split(pool_df, pool_df[target_col], groups=pool_df[patient_col])):
        fold_assign[val_idx] = fold_id
    return fold_assign

pool_df["_fold"] = make_or_reuse_folds(pool_df)

# kiểm tra không rò rỉ bệnh nhân giữa các fold
_leak = (pool_df.groupby(PATIENT_COL)["_fold"].nunique() > 1)
assert not _leak.any(), f"Rò rỉ bệnh nhân giữa các fold: {_leak[_leak].index.tolist()[:5]}..."
print(pool_df["_fold"].value_counts().sort_index())

Dùng lại cột fold có sẵn 'fold' -- khớp fold của baseline from-scratch.
_fold
0    3041
1    3045
2    3061
3    3062
4    3072
Name: count, dtype: int64


## Bước 4 — Backbone + head STEMI + freeze/unfreeze (6 mức)

Backbone **phải cùng kiến trúc/tham số** với notebook 14 (ResNet1D) để `load_state_dict()` khớp.
6 mức freeze/unfreeze theo đúng thang đã dùng cho ECGFounder ở Phần I: từ chỉ train head (mức 0)
đến full fine-tune (mức 5).

In [6]:
class ResidualBlock1D(nn.Module):
    """Verbatim từ stemi_pipeline_optimized_v3.ipynb mục 11 -- PHẢI khớp notebook 14."""
    def __init__(self, c_in, c_out, k=7, stride=2, dropout=0.1):
        super().__init__()
        self.conv1 = nn.Conv1d(c_in, c_out, k, stride, k // 2, bias=False)
        self.bn1 = nn.BatchNorm1d(c_out)
        self.conv2 = nn.Conv1d(c_out, c_out, k, 1, k // 2, bias=False)
        self.bn2 = nn.BatchNorm1d(c_out)
        self.drop = nn.Dropout(dropout)
        self.relu = nn.ReLU(inplace=True)
        self.short = (nn.Identity() if (stride == 1 and c_in == c_out)
                      else nn.Sequential(nn.Conv1d(c_in, c_out, 1, stride, bias=False),
                                         nn.BatchNorm1d(c_out)))

    def forward(self, x):
        idt = self.short(x)
        out = self.drop(self.relu(self.bn1(self.conv1(x))))
        return self.relu(self.bn2(self.conv2(out)) + idt)

class ResNet1DBackbone(nn.Module):
    def __init__(self, channels=(32, 64, 128, 256), in_ch=NUM_LEADS):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv1d(in_ch, 32, 15, 2, 7, bias=False),
                                  nn.BatchNorm1d(32), nn.ReLU(inplace=True), nn.MaxPool1d(2))
        blocks, c_in = [], 32
        for c_out in channels:
            blocks.append(ResidualBlock1D(c_in, c_out, stride=2))
            c_in = c_out
        self.blocks = nn.Sequential(*blocks)
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.out_dim = c_in

    def forward(self, x):
        x = self.blocks(self.stem(x))
        return self.pool(x).squeeze(-1)

class BinaryHead(nn.Module):
    def __init__(self, in_dim):
        super().__init__()
        self.fc = nn.Linear(in_dim, 1)
    def forward(self, x):
        return self.fc(x).squeeze(-1)

class FinetuneModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = ResNet1DBackbone()
        self.head = BinaryHead(self.backbone.out_dim)
    def forward(self, x):
        return self.head(self.backbone(x))

def load_pretrained_backbone(model, ckpt_path=BACKBONE_CKPT_PATH):
    # weights_only=False: checkpoint tự tạo (notebook 14), đáng tin cậy -- mặc định
    # weights_only=True của torch>=2.6 chặn cả state_dict thuần tensor một cách không cần thiết.
    state = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    missing, unexpected = model.backbone.load_state_dict(state, strict=True)
    return model

FREEZE_LEVELS = {
    0: "freeze_all",         # chỉ train head (linear probe)
    1: "unfreeze_block4",    # + block cuối
    2: "unfreeze_block34",   # + 2 block cuối
    3: "unfreeze_block234",  # + 3 block cuối
    4: "unfreeze_all_blocks",# toàn bộ blocks, giữ nguyên stem
    5: "full_finetune",      # toàn bộ backbone + stem
}

def apply_freeze_level(model, level):
    assert level in FREEZE_LEVELS, f"level phải trong {list(FREEZE_LEVELS)}"
    for p in model.backbone.parameters():
        p.requires_grad = False
    for p in model.head.parameters():
        p.requires_grad = True
    n_blocks = len(model.backbone.blocks)
    if level >= 1:
        for p in model.backbone.blocks[n_blocks - 1].parameters(): p.requires_grad = True
    if level >= 2:
        for p in model.backbone.blocks[n_blocks - 2].parameters(): p.requires_grad = True
    if level >= 3:
        for p in model.backbone.blocks[n_blocks - 3].parameters(): p.requires_grad = True
    if level >= 4:
        for blk in model.backbone.blocks:
            for p in blk.parameters(): p.requires_grad = True
    if level >= 5:
        for p in model.backbone.stem.parameters(): p.requires_grad = True
    return FREEZE_LEVELS[level]

_sanity_model = FinetuneModel()
load_pretrained_backbone(_sanity_model)
_x = torch.randn(2, NUM_LEADS, SIGNAL_LEN)
assert _sanity_model(_x).shape == (2,)
for _lvl in range(6):
    apply_freeze_level(_sanity_model, _lvl)
n_trainable = sum(p.numel() for p in _sanity_model.parameters() if p.requires_grad)
print(f"Load backbone pretrain + sanity check OK. Mức 5 (full): {n_trainable:,} tham số trainable.")
del _sanity_model

Load backbone pretrain + sanity check OK. Mức 5 (full): 970,497 tham số trainable.


## Bước 5 — Dataset & training loop (dùng chung cho screening + full K-fold)

**Tier 1:** thêm 5 phép augmentation khi train (verbatim `stemi_pipeline_optimized_v3.ipynb`
mục 9: time-shift, nhiễu Gaussian, gain jitter, nhiễu điện lưới, lead-dropout), Focal Loss +
label smoothing thay BCE trơn (bù mất cân bằng lớp dương ~6-8%), warmup tuyến tính +
`ReduceLROnPlateau` + early stopping thay vì train cố định n_epochs. Checkpoint đổi sang
`RUN_TAG_SUFFIX="_tier1"` nên sẽ train lại từ đầu (từ backbone pretrain), không đụng vào
checkpoint Tier 0 đã có.

In [7]:
from torch.optim.lr_scheduler import ReduceLROnPlateau


def _time_shift(x: np.ndarray, max_shift: int) -> np.ndarray:
    shift = np.random.randint(-max_shift, max_shift + 1)
    if shift == 0:
        return x
    if shift > 0:
        return np.pad(x, ((0, 0), (shift, 0)), mode="edge")[:, :x.shape[1]]
    return np.pad(x, ((0, 0), (0, -shift)), mode="edge")[:, -x.shape[1]:]


def _powerline_noise(x: np.ndarray, fs: int, hz: float, std: float) -> np.ndarray:
    """Nhiễu điện lưới chồng lên MỌI chuyển đạo cùng lúc, biên độ/pha ngẫu nhiên mỗi mẫu --
    verbatim stemi_pipeline_optimized_v3.ipynb mục 9."""
    t = np.arange(x.shape[1]) / fs
    phase = np.random.uniform(0, 2 * np.pi)
    amp = np.random.uniform(0.3, 1.0) * std
    noise = (amp * np.sin(2 * np.pi * hz * t + phase)).astype(np.float32)
    return x + noise[None, :]


def _lead_dropout(x: np.ndarray) -> np.ndarray:
    """Mô phỏng một chuyển đạo bị rớt/tiếp xúc kém -- verbatim mục 9."""
    x = x.copy()
    lead = np.random.randint(0, x.shape[0])
    x[lead] = np.random.normal(0.0, 0.05, size=x.shape[1]).astype(np.float32)
    return x


def augment_ecg(x: np.ndarray) -> np.ndarray:
    """5 phép augment của stemi_pipeline_optimized_v3.ipynb mục 9."""
    if np.random.rand() < AUG_OP_PROB:
        x = _time_shift(x, AUG_MAX_SHIFT)
    if np.random.rand() < AUG_OP_PROB:
        x = x + np.random.normal(0.0, AUG_NOISE_STD, size=x.shape).astype(np.float32)
    if np.random.rand() < AUG_OP_PROB:
        gain = np.random.uniform(1 - AUG_GAIN_JITTER, 1 + AUG_GAIN_JITTER,
                                 size=(NUM_LEADS, 1)).astype(np.float32)
        x = x * gain
    if np.random.rand() < AUG_POWERLINE_PROB:
        x = _powerline_noise(x, FS, AUG_POWERLINE_HZ, AUG_POWERLINE_STD)
    if np.random.rand() < AUG_LEAD_DROPOUT_PROB:
        x = _lead_dropout(x)
    return x


class FocalLossWithSmoothing(nn.Module):
    """BCE có trọng số alpha (theo tỷ lệ lớp thật) + điều chỉnh focal gamma (tập trung ca khó)
    + label smoothing -- verbatim stemi_pipeline_optimized_v3.ipynb mục 12."""

    def __init__(self, alpha_pos: float, gamma: float = 2.0, label_smoothing: float = 0.0):
        super().__init__()
        assert 0.0 < alpha_pos < 1.0
        self.alpha_pos = float(alpha_pos)
        self.gamma = float(gamma)
        self.eps = float(label_smoothing)

    def forward(self, logits, targets):
        targets = targets.float()
        targets_smooth = targets * (1 - self.eps) + self.eps / 2 if self.eps > 0 else targets
        bce = F.binary_cross_entropy_with_logits(logits, targets_smooth, reduction="none")
        if self.gamma <= 0 and self.alpha_pos == 0.5:
            return bce.mean()
        with torch.no_grad():
            p = torch.sigmoid(logits)
            p_t = p * targets + (1 - p) * (1 - targets)
            alpha_t = self.alpha_pos * targets + (1 - self.alpha_pos) * (1 - targets)
            focal_w = alpha_t * (1 - p_t).clamp(min=1e-6, max=1.0) ** self.gamma
        return (focal_w * bce).mean()


class ECGBinaryDataset(Dataset):
    def __init__(self, df, cache_array, target_col=TARGET_COL, augment=False):
        self.df = df.reset_index(drop=True)
        self.cache = cache_array
        self.target_col = target_col
        self.augment = augment and AUG_ENABLE

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        x = np.asarray(self.cache[row["_cache_idx"]], dtype=np.float32)
        if self.augment:
            x = augment_ecg(x)
        y = np.float32(row[self.target_col])
        return torch.from_numpy(np.ascontiguousarray(x)), torch.tensor(y)


def save_checkpoint(model, optimizer, epoch, val_auprc, path):
    torch.save({"epoch": epoch, "model_state": model.state_dict(),
               "optimizer_state": optimizer.state_dict(), "val_auprc": val_auprc}, path)


def load_checkpoint_if_exists(model, optimizer, path):
    if path.exists():
        # weights_only=False: checkpoint tự tạo trong chính notebook này (chứa val_auprc kiểu
        # numpy scalar) -- mặc định weights_only=True của torch>=2.6 sẽ raise UnpicklingError.
        ckpt = torch.load(path, map_location=DEVICE, weights_only=False)
        model.load_state_dict(ckpt["model_state"])
        optimizer.load_state_dict(ckpt["optimizer_state"])
        print(f"Resume từ checkpoint epoch={ckpt['epoch']}, val_auprc={ckpt['val_auprc']:.4f}")
        return ckpt["epoch"] + 1
    return 0


def _apply_warmup_lr(optimizer, base_lr, epoch):
    """Warmup tuyến tính WARMUP_EPOCHS epoch đầu -- verbatim stemi_pipeline_optimized_v3.ipynb.
    Trả về True nếu vẫn đang trong giai đoạn warmup (scheduler CHƯA được can thiệp)."""
    if WARMUP_EPOCHS <= 0 or epoch >= WARMUP_EPOCHS:
        return False
    for g in optimizer.param_groups:
        g["lr"] = base_lr * (epoch + 1) / WARMUP_EPOCHS
    return epoch < WARMUP_EPOCHS - 1


def train_one_run(train_loader, val_loader, freeze_level, run_tag, n_epochs=None):
    """Train 1 model (1 fold hoặc screening) với 1 mức freeze/unfreeze, trả về best val AUPRC
    + đường dẫn checkpoint tốt nhất + xác suất dự đoán OOF trên val_loader (dùng checkpoint tốt
    nhất). Tier 1: Focal Loss + label smoothing (bù mất cân bằng lớp dương), warmup +
    ReduceLROnPlateau (ổn định hội tụ), early stopping (dừng đúng lúc thay vì train cố định
    n_epochs), grad clipping -- cùng bộ kỹ thuật stemi_pipeline_optimized_v3.ipynb đã dùng."""
    model = FinetuneModel().to(DEVICE)
    load_pretrained_backbone(model)
    apply_freeze_level(model, freeze_level)
    trainable_params = [p for p in model.parameters() if p.requires_grad]
    base_lr = 3e-4
    optimizer = torch.optim.AdamW(trainable_params, lr=base_lr, weight_decay=1e-4)
    scheduler = ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=LR_PATIENCE)

    alpha_pos = 1.0 - float(train_loader.dataset.df[train_loader.dataset.target_col].mean())
    criterion = FocalLossWithSmoothing(alpha_pos=alpha_pos, gamma=FOCAL_GAMMA,
                                       label_smoothing=LABEL_SMOOTH_EPS)

    ckpt_path = FINETUNE_MODELS_DIR / f"{run_tag}_latest.pt"
    best_path = FINETUNE_MODELS_DIR / f"{run_tag}_best.pt"
    start_epoch = load_checkpoint_if_exists(model, optimizer, ckpt_path)

    n_epochs = n_epochs or (2 if RUN_MODE == "debug" else 30)
    best_auprc = -1.0
    best_y_true = best_y_prob = None
    last_y_true = last_y_prob = None
    bad_epochs = 0

    for epoch in range(start_epoch, n_epochs):
        in_warmup = _apply_warmup_lr(optimizer, base_lr, epoch)
        model.train()
        for x, y in train_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(x), y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

        model.eval()
        y_true_all, y_prob_all = [], []
        with torch.no_grad():
            for x, y in val_loader:
                x = x.to(DEVICE)
                prob = torch.sigmoid(model(x)).cpu().numpy()
                y_true_all.append(y.numpy())
                y_prob_all.append(prob)
        y_true_all = np.concatenate(y_true_all)
        y_prob_all = np.concatenate(y_prob_all)
        auprc = average_precision_score(y_true_all, y_prob_all) if y_true_all.sum() > 0 else float("nan")
        if not in_warmup:
            scheduler.step(auprc if not np.isnan(auprc) else -np.inf)

        print(f"  [{run_tag}] epoch {epoch + 1}/{n_epochs} -- val_AUPRC={auprc:.4f} "
              f"lr={optimizer.param_groups[0]['lr']:.2e}")
        last_y_true, last_y_prob = y_true_all, y_prob_all
        save_checkpoint(model, optimizer, epoch, auprc, ckpt_path)
        if not np.isnan(auprc) and auprc > best_auprc:
            best_auprc = auprc
            bad_epochs = 0
            save_checkpoint(model, optimizer, epoch, auprc, best_path)
            best_y_true, best_y_prob = y_true_all, y_prob_all
        else:
            bad_epochs += 1
        if bad_epochs >= EARLY_STOP_PATIENCE:
            print(f"  [{run_tag}] early stop @ epoch {epoch + 1} (không cải thiện "
                  f"{EARLY_STOP_PATIENCE} epoch liên tiếp)")
            break

    if best_y_true is None:
        if start_epoch >= n_epochs and best_path.exists():
            # Checkpoint đã train đủ/early-stop từ MỘT LẦN CHẠY TRƯỚC -- vòng for phía trên
            # không chạy bước nào. best_path trên đĩa đã đúng (checkpoint của epoch tốt nhất
            # thật sự) -- TUYỆT ĐỐI không ghi đè bằng ckpt_path (epoch cuối), chỉ eval lại bằng
            # đúng trọng số best đã lưu để lấy y_true/y_prob cho OOF.
            print(f"  [{run_tag}] Đã hoàn tất từ trước -- eval lại bằng checkpoint tốt nhất đã "
                  f"lưu, không train/ghi đè thêm.")
            _best_state = torch.load(best_path, map_location=DEVICE, weights_only=False)
            model.load_state_dict(_best_state["model_state"])
            model.eval()
            y_true_all, y_prob_all = [], []
            with torch.no_grad():
                for x, y in val_loader:
                    x = x.to(DEVICE)
                    prob = torch.sigmoid(model(x)).cpu().numpy()
                    y_true_all.append(y.numpy())
                    y_prob_all.append(prob)
            best_y_true = np.concatenate(y_true_all)
            best_y_prob = np.concatenate(y_prob_all)
            best_auprc = (average_precision_score(best_y_true, best_y_prob)
                          if best_y_true.sum() > 0 else float(_best_state.get("val_auprc", float("nan"))))
        else:
            # Không epoch nào có AUPRC hợp lệ -- val fold thiếu lớp dương (dễ gặp với target hiếm như
            # STEMI/OMI ở fold nhỏ). Fallback: dùng checkpoint + dự đoán epoch cuối thay vì crash.
            print(f"  [CẢNH BÁO] {run_tag}: không epoch nào có AUPRC hợp lệ (val fold có thể thiếu lớp "
                  f"dương) -- dùng checkpoint epoch cuối làm fallback.")
            shutil.copy2(ckpt_path, best_path)
            best_y_true, best_y_prob = last_y_true, last_y_prob

    return {"best_auprc": best_auprc, "best_ckpt": best_path,
            "oof_y_true": best_y_true, "oof_y_prob": best_y_prob}

## Bước 6 — Giai đoạn A: screening 1-fold qua 6 mức freeze/unfreeze

In [8]:
BATCH_SIZE = 8 if RUN_MODE == "debug" else 64

screen_fold = 0
screen_train_df = pool_df[pool_df["_fold"] != screen_fold]
screen_val_df = pool_df[pool_df["_fold"] == screen_fold]

# pool_cache: mảng waveform đã tiền xử lý, index khớp cột "_cache_idx" -- gán ở Bước 2
# (load_pool_and_test), cùng cache vật lý dùng chung với test_cache.
screen_train_loader = DataLoader(ECGBinaryDataset(screen_train_df, pool_cache, augment=True),
                                  batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
screen_val_loader = DataLoader(ECGBinaryDataset(screen_val_df, pool_cache),
                                batch_size=BATCH_SIZE, shuffle=False)
assert len(screen_train_loader) > 0, "screen_train_loader rỗng -- tăng DEBUG_LIMIT hoặc giảm BATCH_SIZE"

screening_results = {}
for level in range(6):
    print(f"\n=== Screening mức freeze/unfreeze {level} ({FREEZE_LEVELS[level]}) ===")
    res = train_one_run(screen_train_loader, screen_val_loader, level,
                         run_tag=f"{TASK}_screen_lvl{level}{RUN_TAG_SUFFIX}", n_epochs=2 if RUN_MODE == "debug" else 8)
    screening_results[level] = res["best_auprc"]

best_level = max(screening_results, key=screening_results.get)
print("\nKết quả screening:", {FREEZE_LEVELS[k]: round(v, 4) for k, v in screening_results.items()})
print(f"Mức thắng: {best_level} ({FREEZE_LEVELS[best_level]}) -- dùng cho full K-fold ở Bước 7.")


=== Screening mức freeze/unfreeze 0 (freeze_all) ===
Resume từ checkpoint epoch=7, val_auprc=0.4558
  [stemi_screen_lvl0_tier1] Đã hoàn tất từ trước -- eval lại bằng checkpoint tốt nhất đã lưu, không train/ghi đè thêm.

=== Screening mức freeze/unfreeze 1 (unfreeze_block4) ===
Resume từ checkpoint epoch=7, val_auprc=0.5636
  [stemi_screen_lvl1_tier1] Đã hoàn tất từ trước -- eval lại bằng checkpoint tốt nhất đã lưu, không train/ghi đè thêm.

=== Screening mức freeze/unfreeze 2 (unfreeze_block34) ===
Resume từ checkpoint epoch=7, val_auprc=0.6244
  [stemi_screen_lvl2_tier1] Đã hoàn tất từ trước -- eval lại bằng checkpoint tốt nhất đã lưu, không train/ghi đè thêm.

=== Screening mức freeze/unfreeze 3 (unfreeze_block234) ===
Resume từ checkpoint epoch=7, val_auprc=0.6402
  [stemi_screen_lvl3_tier1] Đã hoàn tất từ trước -- eval lại bằng checkpoint tốt nhất đã lưu, không train/ghi đè thêm.

=== Screening mức freeze/unfreeze 4 (unfreeze_all_blocks) ===
Resume từ checkpoint epoch=7, val_auprc

## Bước 7 — Giai đoạn B: full K-fold với mức freeze/unfreeze thắng → OOF toàn POOL

**Cập nhật (hiệu chuẩn Platt):** ngoài `_oof_prob` (thô), giờ có thêm `_oof_prob_cal` (cross-fit
Platt, verbatim `stemi_pipeline_optimized_v3.ipynb` mục 16) và `PLATT_FULL` (hiệu chuẩn triển
khai, fit trên toàn bộ OOF, dùng ở Bước 9 cho TEST). Fix cho lỗi SD Sensitivity giữa fold bị thổi
phồng do Focal Loss (Tier 1) làm lệch thang xác suất thô giữa 5 model fold khác nhau.

In [9]:
from sklearn.linear_model import LogisticRegression

oof_y_prob = np.full(len(pool_df), np.nan)
oof_y_true = pool_df[TARGET_COL].to_numpy().astype(float)

for fold_id in sorted(pool_df["_fold"].unique()):
    print(f"\n=== Fold {fold_id}/{pool_df['_fold'].nunique() - 1} (mức {best_level}: "
          f"{FREEZE_LEVELS[best_level]}) ===")
    tr_df = pool_df[pool_df["_fold"] != fold_id]
    va_df = pool_df[pool_df["_fold"] == fold_id]
    tr_loader = DataLoader(ECGBinaryDataset(tr_df, pool_cache, augment=True), batch_size=BATCH_SIZE,
                            shuffle=True, drop_last=True)
    va_loader = DataLoader(ECGBinaryDataset(va_df, pool_cache), batch_size=BATCH_SIZE, shuffle=False)
    assert len(tr_loader) > 0, f"Fold {fold_id}: train_loader rỗng"

    res = train_one_run(tr_loader, va_loader, best_level, run_tag=f"{TASK}_fold{fold_id}{RUN_TAG_SUFFIX}",
                         n_epochs=2 if RUN_MODE == "debug" else 30)
    oof_y_prob[va_df.index.to_numpy()] = res["oof_y_prob"]

assert not np.isnan(oof_y_prob).any(), "Có record chưa được gán OOF -- kiểm tra lại vòng K-fold"

# =============================================================================
# Hiệu chuẩn Platt -- verbatim stemi_pipeline_optimized_v3.ipynb mục 16.
#
# 5 fold là 5 MODEL KHÁC NHAU (mỗi model chỉ train trên 4/5 POOL) -- Focal Loss (Tier 1) làm
# lệch thang xác suất thô giữa các model này nhiều hơn BCE trơn (alpha_pos khác nhau theo tỷ lệ
# dương từng fold + gamma hạ trọng số ca dễ không đều), khiến SD Sensitivity-tại-1-ngưỡng giữa
# các fold bị thổi phồng và biên an toàn ở Bước 8 ép ngưỡng sai (Sensitivity ảo cao, Specificity
# sụp). Cách sửa: đưa xác suất mọi fold về CÙNG thang trước khi tính SD/khoá ngưỡng.
# =============================================================================
_fold_id_arr = pool_df["_fold"].to_numpy()


def _logit(p, eps=1e-6):
    p = np.clip(np.asarray(p, dtype=np.float64), eps, 1 - eps)
    return np.log(p / (1 - p))


def crossfit_platt(p_raw, y_true, fold_id):
    """Fold k được hiệu chuẩn bằng bộ Platt fit trên OOF của các fold KHÁC -- ước lượng TRUNG
    THỰC (không tự chấm điểm mình), dùng để tính SD Sensitivity giữa fold ở Bước 8."""
    out = np.full(len(p_raw), np.nan)
    for k in np.unique(fold_id):
        fit_mask = fold_id != k
        apply_mask = fold_id == k
        lr = LogisticRegression(C=1e6, solver="lbfgs", max_iter=1000)
        lr.fit(_logit(p_raw[fit_mask]).reshape(-1, 1), y_true[fit_mask].astype(int))
        out[apply_mask] = lr.predict_proba(_logit(p_raw[apply_mask]).reshape(-1, 1))[:, 1]
    return out


oof_y_prob_cal = crossfit_platt(oof_y_prob, oof_y_true, _fold_id_arr)

# PLATT_FULL: fit trên TOÀN BỘ OOF (không cross-fit -- sẽ áp lên TEST, tách biệt hoàn toàn nên
# không có rủi ro rò rỉ) -- đây là bộ hiệu chuẩn "triển khai" dùng ở Bước 9 cho dự đoán TEST.
PLATT_FULL = LogisticRegression(C=1e6, solver="lbfgs", max_iter=1000)
PLATT_FULL.fit(_logit(oof_y_prob).reshape(-1, 1), oof_y_true.astype(int))


def apply_platt(p_raw):
    return PLATT_FULL.predict_proba(_logit(np.asarray(p_raw)).reshape(-1, 1))[:, 1]


pool_df["_oof_prob"] = oof_y_prob
pool_df["_oof_prob_cal"] = oof_y_prob_cal
pool_df[["record_id", TARGET_COL, "_fold", "_oof_prob", "_oof_prob_cal"]].to_csv(
    FINETUNE_OOF_DIR / f"{TASK}_pool_oof.csv", index=False)
print(f"\nOOF AUPRC toàn POOL -- thô: {average_precision_score(oof_y_true, oof_y_prob):.4f} | "
      f"sau hiệu chuẩn: {average_precision_score(oof_y_true, oof_y_prob_cal):.4f} "
      f"(AUPRC không đổi nhiều vì Platt chỉ scale lại xác suất, không đổi thứ hạng)")


=== Fold 0/4 (mức 4: unfreeze_all_blocks) ===
Resume từ checkpoint epoch=24, val_auprc=0.6444
  [stemi_fold0_tier1] epoch 26/30 -- val_AUPRC=0.5871 lr=1.50e-04
  [stemi_fold0_tier1] epoch 27/30 -- val_AUPRC=0.5842 lr=1.50e-04
  [stemi_fold0_tier1] epoch 28/30 -- val_AUPRC=0.5558 lr=1.50e-04
  [stemi_fold0_tier1] epoch 29/30 -- val_AUPRC=0.5974 lr=1.50e-04
  [stemi_fold0_tier1] epoch 30/30 -- val_AUPRC=0.6261 lr=1.50e-04

=== Fold 1/4 (mức 4: unfreeze_all_blocks) ===
Resume từ checkpoint epoch=20, val_auprc=0.6630
  [stemi_fold1_tier1] epoch 22/30 -- val_AUPRC=0.6286 lr=1.50e-04
  [stemi_fold1_tier1] epoch 23/30 -- val_AUPRC=0.6473 lr=1.50e-04
  [stemi_fold1_tier1] epoch 24/30 -- val_AUPRC=0.6655 lr=1.50e-04
  [stemi_fold1_tier1] epoch 25/30 -- val_AUPRC=0.6616 lr=1.50e-04
  [stemi_fold1_tier1] epoch 26/30 -- val_AUPRC=0.6318 lr=1.50e-04
  [stemi_fold1_tier1] epoch 27/30 -- val_AUPRC=0.6370 lr=1.50e-04
  [stemi_fold1_tier1] epoch 28/30 -- val_AUPRC=0.6102 lr=1.50e-04
  [stemi_fold1_tie

## Bước 8 — Khoá threshold trên POOL-OOF (Sensitivity ≥91% làm sàn + biên an toàn)

Dùng `threshold_at_sensitivity` (dò trực tiếp trên điểm số các ca dương, verbatim
`stemi_pipeline_optimized_v3.ipynb` mục 19) rồi áp **biên an toàn `SENS_MARGIN_Z`**: đo SD
của Sensitivity giữa 5 fold tại ngưỡng ứng với mục tiêu, nhắm cao hơn `z=1.28` lần SD đó trước
khi khoá — chống đúng lỗi "khoá đúng % trên POOL nhưng tụt hẳn trên TEST" mà lần chạy trước
của chính notebook này gặp phải (91% POOL → 76.5% TEST).

**Cập nhật:** khoá ngưỡng trên `oof_y_prob_cal` (đã hiệu chuẩn Platt ở Bước 7) thay vì xác suất
thô -- cell in thêm dòng chẩn đoán so sánh SD Sensitivity CÓ/KHÔNG hiệu chuẩn để thấy rõ mức sửa.

In [10]:
def threshold_at_sensitivity(y_true, y_prob, target_sens):
    """Ngưỡng LỚN NHẤT sao cho Sensitivity thực tế vẫn >= target_sens -- dò trực tiếp trên điểm
    số của các ca dương (không nội suy tuyến tính trên ROC dạng bậc thang, vốn có thể cho kết
    quả tụt dưới mục tiêu). Verbatim stemi_pipeline_optimized_v3.ipynb mục 19."""
    y_true = np.asarray(y_true).astype(int)
    pos_scores = np.sort(np.asarray(y_prob)[y_true == 1])[::-1]
    n_pos = len(pos_scores)
    k = min(int(np.ceil(target_sens * n_pos)), n_pos)
    return float(pos_scores[k - 1]) if k > 0 else 1.0

def lock_threshold_youden(y_true, y_prob):
    fpr, tpr, thresholds = roc_curve(y_true, y_prob)
    j = tpr - fpr
    return float(thresholds[np.argmax(j)])

def compute_metrics(y_true, y_prob, threshold):
    y_pred = (y_prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return {
        "sensitivity": tp / (tp + fn) if (tp + fn) else float("nan"),
        "specificity": tn / (tn + fp) if (tn + fp) else float("nan"),
        "ppv": tp / (tp + fp) if (tp + fp) else float("nan"),
        "npv": tn / (tn + fn) if (tn + fn) else float("nan"),
        "tp": int(tp), "fp": int(fp), "fn": int(fn), "tn": int(tn),
    }

# =============================================================================
# Chốt ngưỡng CÓ BIÊN AN TOÀN -- verbatim cơ chế mục 19 stemi_pipeline_optimized_v3.ipynb.
#
# Vấn đề đang sửa: ngưỡng chốt trên POOL-OOF để đạt Sensitivity = S gần như KHÔNG BAO GIỜ cho
# đúng S trên TEST -- TEST chỉ có vài trăm ca dương nên Sensitivity ở đó dao động thuần do cỡ
# mẫu. Lần chạy trước của chính notebook này rơi đúng vào chiều xấu (xem phân tích kết quả):
# khoá đúng mục tiêu trên POOL nhưng Sensitivity trên TEST tụt hàng chục điểm % -- vi phạm
# chính sách lâm sàng đặt ra. stemi_pipeline_optimized_v3.ipynb từng dính lỗi y hệt ở v2
# (chốt 91% trên POOL -> chỉ đạt 88,9% trên TEST) và đã vá bằng cơ chế dưới đây.
#
# Cách sửa (thuần POOL, không nhìn TEST): tại đúng ngưỡng ứng với mục tiêu, đo Sensitivity đạt
# được trong TỪNG fold rồi lấy SD giữa các fold (mỗi fold cỡ tương đương TEST, nên SD này ước
# lượng đúng mức dao động sẽ gặp khi chuyển sang TEST). Nhắm cao hơn mục tiêu z=1.28 lần SD đó
# để xác suất đạt mục tiêu thật trên TEST ~90%. Trần SENS_MARGIN_MAX_TARGET chặn trường hợp SD
# lớn bất thường đẩy mục tiêu sát 1.0 (ngưỡng tụt xuống dự đoán MỌI ca dương, Specificity=0 --
# hỏng âm thầm mà vẫn in kết quả bình thường).
# =============================================================================

SENS_MARGIN_Z = 1.28            # verbatim stemi_pipeline_optimized_v3.ipynb
SENS_MARGIN_MAX_TARGET = 0.97   # trần bắt buộc

def fold_sensitivity_sd(y_true, y_prob, fold_id, thr):
    """SD của Sensitivity giữa các fold tại CÙNG một ngưỡng."""
    sens = []
    for k in np.unique(fold_id):
        m = fold_id == k
        yk, pk = y_true[m], y_prob[m]
        if yk.sum() == 0:
            continue
        sens.append(float((pk[yk == 1] >= thr).mean()))
    return float(np.std(sens, ddof=1)) if len(sens) > 1 else 0.0

def threshold_with_margin(y_true, y_prob, fold_id, target, z=SENS_MARGIN_Z):
    t_plain = threshold_at_sensitivity(y_true, y_prob, target)
    sd = fold_sensitivity_sd(y_true, y_prob, fold_id, t_plain)
    raw_adj = target + z * sd
    target_adj = float(min(raw_adj, SENS_MARGIN_MAX_TARGET))
    capped = raw_adj > SENS_MARGIN_MAX_TARGET + 1e-12
    return threshold_at_sensitivity(y_true, y_prob, target_adj), target_adj, sd, capped

if THRESHOLD_POLICY == "sensitivity_floor":
    _base_target = SENSITIVITY_FLOOR
elif THRESHOLD_POLICY == "youden":
    # Youden chỉ cho 1 điểm cân bằng, không có "sàn" để bảo vệ như sensitivity_floor -- dùng
    # chính Sensitivity mà điểm cân bằng đó đạt trên POOL-OOF (đã hiệu chuẩn) làm mục tiêu cần
    # bảo vệ khi chuyển sang TEST, rồi áp CÙNG cơ chế biên an toàn bên trên.
    _youden_thr = lock_threshold_youden(oof_y_true, oof_y_prob_cal)
    _base_target = compute_metrics(oof_y_true, oof_y_prob_cal, _youden_thr)["sensitivity"]
else:
    raise ValueError(f"THRESHOLD_POLICY không hợp lệ: {THRESHOLD_POLICY}")

# Chẩn đoán: SD Sensitivity nếu KHÔNG hiệu chuẩn (Bước 7) -- để thấy rõ mức cải thiện.
_diag_thr, _diag_target_adj, _diag_sd, _diag_capped = threshold_with_margin(
    oof_y_true, oof_y_prob, pool_df["_fold"].to_numpy(), _base_target)
print(f"(Chẩn đoán, không dùng) SD Sensitivity trên xác suất THÔ (chưa hiệu chuẩn): "
      f"{_diag_sd:.4f} -> nếu không hiệu chuẩn, mục tiêu sẽ bị nâng lên {_diag_target_adj:.4f}")

LOCKED_THRESHOLD, _target_adj, _fold_sd, _capped = threshold_with_margin(
    oof_y_true, oof_y_prob_cal, pool_df["_fold"].to_numpy(), _base_target)

_oof_metrics = compute_metrics(oof_y_true, oof_y_prob_cal, LOCKED_THRESHOLD)
print(f"Threshold khoá trên POOL-OOF đã hiệu chuẩn ({THRESHOLD_POLICY}, biên an toàn z={SENS_MARGIN_Z}): "
      f"{LOCKED_THRESHOLD:.4f}")
print(f"  Mục tiêu gốc: {_base_target:.4f} | SD Sensitivity giữa 5 fold (đã hiệu chuẩn): {_fold_sd:.4f} | "
      f"mục tiêu đã nâng: {_target_adj:.4f}" + (" -- ĐÃ CHẠM TRẦN AN TOÀN" if _capped else ""))
print(json.dumps(_oof_metrics, indent=2))

(Chẩn đoán, không dùng) SD Sensitivity trên xác suất THÔ (chưa hiệu chuẩn): 0.0271 -> nếu không hiệu chuẩn, mục tiêu sẽ bị nâng lên 0.9447
Threshold khoá trên POOL-OOF đã hiệu chuẩn (sensitivity_floor, biên an toàn z=1.28): 0.0190
  Mục tiêu gốc: 0.9100 | SD Sensitivity giữa 5 fold (đã hiệu chuẩn): 0.0325 | mục tiêu đã nâng: 0.9516
{
  "sensitivity": 0.9518367346938775,
  "specificity": 0.6473392145702903,
  "ppv": 0.19042952800914584,
  "npv": 0.9935575453155711,
  "tp": 1166,
  "fp": 4957,
  "fn": 59,
  "tn": 9099
}


## Bước 9 — Train FINAL trên toàn POOL, đánh giá TEST bằng Bagging (chỉ chạm 1 lần, ở đây)

Ngoài train FINAL (retrain trên toàn POOL) như cũ, giờ còn **bagging**: trung bình xác suất của
5 checkpoint fold đã lưu sẵn ở Bước 7 (không train thêm) — verbatim thiết kế
`stemi_pipeline_optimized_v3.ipynb` (`PRIMARY_PREDICTOR="bagging"`, đo được +0.03..+0.05 AUPRC
so với FINAL). Quyết định dùng bagging làm predictor chính được **cố định trước khi nhìn TEST**
(không chọn theo điểm TEST) — FINAL vẫn được tính để đối chiếu, in ra tham khảo.

In [11]:
full_pool_loader = DataLoader(ECGBinaryDataset(pool_df, pool_cache, augment=True), batch_size=BATCH_SIZE,
                              shuffle=True, drop_last=True)
# Dùng screen_val_loader làm "validation" theo dõi trong lúc train FINAL (không dùng để chọn gì
# trên TEST) -- có thể thay bằng 1 phần nhỏ POOL trích riêng nếu muốn tách bạch hơn.
final_res = train_one_run(full_pool_loader, screen_val_loader, best_level,
                           run_tag=f"{TASK}_FINAL{RUN_TAG_SUFFIX}", n_epochs=2 if RUN_MODE == "debug" else 30)

# test_cache: cùng mảng memmap với pool_cache (gán ở Bước 2), index khớp cột "_cache_idx".
test_loader = DataLoader(ECGBinaryDataset(test_df, test_cache), batch_size=BATCH_SIZE, shuffle=False)


def predict_on_loader(ckpt_path, loader):
    m = FinetuneModel().to(DEVICE)
    # weights_only=False: checkpoint tự tạo trong chính notebook này (chứa val_auprc kiểu numpy
    # scalar) -- mặc định weights_only=True của torch>=2.6 sẽ raise UnpicklingError.
    ckpt = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
    m.load_state_dict(ckpt["model_state"])
    m.eval()
    y_true_, y_prob_ = [], []
    with torch.no_grad():
        for x, y in loader:
            x = x.to(DEVICE)
            y_prob_.append(torch.sigmoid(m(x)).cpu().numpy())
            y_true_.append(y.numpy())
    return np.concatenate(y_true_), np.concatenate(y_prob_)


test_y_true, _final_y_prob_raw = predict_on_loader(final_res["best_ckpt"], test_loader)
# Áp CÙNG bộ hiệu chuẩn Platt (PLATT_FULL, fit trên OOF ở Bước 7) lên xác suất thô trên TEST --
# threshold đã khoá ở Bước 8 nằm trên thang ĐÃ hiệu chuẩn, nên dự đoán TEST cũng phải hiệu
# chuẩn để cùng thang, nếu không ngưỡng sẽ lệch hệt như lỗi đã chẩn đoán (Sensitivity ảo cao).
final_y_prob = apply_platt(_final_y_prob_raw)

# --- Bagging: trung bình xác suất của 5 checkpoint fold ĐÃ CÓ SẴN từ Bước 7 (không train thêm)
# -- stemi_pipeline_optimized_v3.ipynb mặc định PRIMARY_PREDICTOR="bagging" vì đo được
# +0.03..+0.05 AUPRC so với 1 model FINAL retrain trên toàn POOL. Quyết định CỐ ĐỊNH trước khi
# nhìn TEST (không chọn theo điểm TEST, tránh rò rỉ lựa chọn model vào TEST).
PRIMARY_PREDICTOR = "bagging"
_fold_ids = sorted(pool_df["_fold"].unique())
_bag_probs = []
for fold_id in _fold_ids:
    _ckpt_path = FINETUNE_MODELS_DIR / f"{TASK}_fold{fold_id}{RUN_TAG_SUFFIX}_best.pt"
    _yt, _yp_raw = predict_on_loader(_ckpt_path, test_loader)
    assert np.array_equal(_yt, test_y_true), f"Thứ tự TEST lệch giữa fold {fold_id} và FINAL"
    _bag_probs.append(apply_platt(_yp_raw))   # hiệu chuẩn TỪNG fold trước khi trung bình
bag_y_prob = np.mean(_bag_probs, axis=0)

_auprc_final = average_precision_score(test_y_true, final_y_prob)
_auprc_bag = average_precision_score(test_y_true, bag_y_prob)
print(f"(Tham khảo, KHÔNG dùng để quyết định) FINAL AUPRC={_auprc_final:.4f} | "
      f"Bagging ({len(_fold_ids)} fold) AUPRC={_auprc_bag:.4f} | chênh {_auprc_bag - _auprc_final:+.4f}")
print(f"Predictor chính dùng cho kết quả cuối: {PRIMARY_PREDICTOR.upper()}")

test_y_prob = bag_y_prob

test_metrics = compute_metrics(test_y_true, test_y_prob, LOCKED_THRESHOLD)
test_auprc = average_precision_score(test_y_true, test_y_prob)
test_auroc = roc_auc_score(test_y_true, test_y_prob) if len(set(test_y_true)) > 1 else float("nan")
print(f"TEST ({PRIMARY_PREDICTOR}) -- AUPRC={test_auprc:.4f} AUROC={test_auroc:.4f}")
print(json.dumps(test_metrics, indent=2))

pd.DataFrame({"record_id": test_df["record_id"], TARGET_COL: test_y_true,
              "y_prob_final": final_y_prob, "y_prob_bagging": bag_y_prob,
              "y_prob": test_y_prob}).to_csv(FINETUNE_OOF_DIR / f"{TASK}_test_predictions.csv", index=False)

Resume từ checkpoint epoch=29, val_auprc=0.9385
  [stemi_FINAL_tier1] Đã hoàn tất từ trước -- eval lại bằng checkpoint tốt nhất đã lưu, không train/ghi đè thêm.
(Tham khảo, KHÔNG dùng để quyết định) FINAL AUPRC=0.6662 | Bagging (5 fold) AUPRC=0.7097 | chênh +0.0435
Predictor chính dùng cho kết quả cuối: BAGGING
TEST (bagging) -- AUPRC=0.7097 AUROC=0.9438
{
  "sensitivity": 0.9769585253456221,
  "specificity": 0.6280487804878049,
  "ppv": 0.1881100266193434,
  "npv": 0.9967741935483871,
  "tp": 212,
  "fp": 915,
  "fn": 5,
  "tn": 1545
}


## Bước 10 — So sánh bootstrap với baseline from-scratch

`stemi_pipeline_optimized_v3.ipynb` (mục 22) lưu xác suất dự đoán TEST của **từng model đơn**
(`bag_{model}`/`fin_{model}`) vào `.../ACS-ECG-AI/outputs/results/stemi_test_predictions_stemi_optimized_v3.npz`
— nhưng **model chính (champion) luôn là một ensemble** có trọng số fit lúc chạy (Greedy/Stacking/
TopK), không được lưu ra file, nên không thể tự động tái tạo chính xác 100% ở đây.

Cell dưới tự tìm và liệt kê các cột có sẵn trong file `.npz` đó (nếu tìm thấy) — chỉ cần điền
đúng 1 dòng chọn cột rồi chạy lại. Muốn so đúng với champion, quay lại notebook gốc, chạy đến hết
mục 16 rồi lưu thêm `champion_test_prob = ENS_APPLY[PRIMARY_ENSEMBLE][1](...)` áp lên TEST_P của
từng model (mục 18) ra một cột riêng.

**Cập nhật:** mặc định tự lấy cột `bag_ResNet1D` (cùng kiến trúc backbone, cùng predictor
bagging, from-scratch) làm `baseline_y_prob` — so sánh 1-model-vs-1-model, tách bạch hiệu ứng
pretrain khỏi hiệu ứng ensemble 8 kiến trúc. Muốn so với ensemble champion thật vẫn cần làm thủ
công theo hướng dẫn ở trên.

In [12]:
def bootstrap_delta_auprc(y_true, y_prob_a, y_prob_b, n_boot=2000, seed=SEED):
    """A=pretrain-init (model ở đây), B=baseline from-scratch. Trả về delta điểm ước lượng, CI 95%,
    p-value xấp xỉ, và có ý nghĩa thống kê hay không (CI không chứa 0)."""
    rng = np.random.default_rng(seed)
    n = len(y_true)
    deltas = np.empty(n_boot)
    for b in range(n_boot):
        idx = rng.integers(0, n, size=n)
        yt = y_true[idx]
        if yt.sum() == 0 or yt.sum() == n:
            idx = rng.integers(0, n, size=n)
            yt = y_true[idx]
        deltas[b] = (average_precision_score(yt, y_prob_a[idx])
                     - average_precision_score(yt, y_prob_b[idx]))
    point = average_precision_score(y_true, y_prob_a) - average_precision_score(y_true, y_prob_b)
    ci_lo, ci_hi = np.percentile(deltas, [2.5, 97.5])
    p_value = 2 * min((deltas <= 0).mean(), (deltas >= 0).mean())
    return {"point_delta_auprc": point, "ci95_lo": ci_lo, "ci95_hi": ci_hi,
            "p_value_approx": p_value, "significant": not (ci_lo <= 0 <= ci_hi)}


def find_baseline_predictions_npz():
    """Tìm file .npz xác suất TEST đã lưu bởi stemi_pipeline_optimized_v3.ipynb (mục 22).
    Không có quyền chọn cột thay bạn (champion là ensemble, không nằm trong file) -- chỉ giúp
    tìm file + liệt kê cột sẵn có."""
    results_dir = DATA_DRIVE_PROJECT / "outputs" / "results"
    if not results_dir.exists():
        print(f"Không thấy {results_dir} -- chạy stemi_pipeline_optimized_v3.ipynb (mục 22) trước, "
              f"hoặc điền thủ công đường dẫn npz vào biến bên dưới.")
        return None
    candidates = sorted(results_dir.glob("stemi_test_predictions_*.npz"))
    if not candidates:
        print(f"Không thấy stemi_test_predictions_*.npz trong {results_dir}.")
        return None
    return candidates[-1]


def load_baseline_column(npz_path, column, test_df=test_df):
    """Nạp 1 cột xác suất từ .npz baseline, sắp lại đúng thứ tự record_id của test_df ở đây --
    thứ tự TEST giữa 2 notebook có thể khác nhau dù cùng tập bệnh nhân."""
    data = np.load(npz_path, allow_pickle=True)
    by_record = dict(zip(data["record_stem"].astype(str), data[column]))
    missing = [r for r in test_df["record_id"] if r not in by_record]
    assert not missing, f"{len(missing)} record_id của TEST ở đây không có trong npz baseline: {missing[:5]}..."
    return np.array([by_record[r] for r in test_df["record_id"]])


_baseline_npz = find_baseline_predictions_npz()
baseline_y_prob = None
_baseline_col = "bag_ResNet1D"   # cùng kiến trúc backbone với pretrain-init (ResNet1D), cùng
                                    # predictor bagging -- so sánh 1-model-vs-1-model, tách bạch
                                    # hiệu ứng pretrain khỏi hiệu ứng ensemble 8 kiến trúc.
if _baseline_npz is not None:
    _keys = [k for k in np.load(_baseline_npz).files if k.startswith(("bag_", "fin_"))]
    print(f"Tìm thấy: {_baseline_npz}")
    if _baseline_col in _keys:
        baseline_y_prob = load_baseline_column(_baseline_npz, _baseline_col)
        print(f"Dùng cột '{_baseline_col}' (cùng kiến trúc, cùng predictor bagging, from-scratch) "
              f"để so sánh công bằng -- tách bạch hiệu ứng pretrain khỏi hiệu ứng ensemble 8 model.")
    else:
        print(f"Không thấy cột '{_baseline_col}' trong npz -- cột có sẵn ({len(_keys)}): {_keys}")
        print('  baseline_y_prob = load_baseline_column(_baseline_npz, "bag_<tên_model>")')

if baseline_y_prob is not None:
    comparison = bootstrap_delta_auprc(test_y_true, test_y_prob, np.asarray(baseline_y_prob))
    print(json.dumps({k: (float(v) if hasattr(v, "item") else v) for k, v in comparison.items()},
                     indent=2))
    if comparison["significant"] and comparison["point_delta_auprc"] > 0:
        print("\n>>> Pretrain-init CẢI THIỆN có ý nghĩa thống kê so với baseline -- cân nhắc đưa vào FINAL/ensemble.")
    elif comparison["significant"] and comparison["point_delta_auprc"] < 0:
        print("\n>>> Pretrain-init KÉM HƠN baseline có ý nghĩa thống kê -- giữ nguyên from-scratch.")
    else:
        print("\n>>> Chênh lệch không có ý nghĩa thống kê -- không đủ bằng chứng để đổi.")
else:
    print("Điền baseline_y_prob trước khi kết luận -- xem gợi ý ở trên (tuỳ chọn, không chặn chạy full).")

Tìm thấy: /content/drive/MyDrive/ACS-ECG-AI/outputs/results/stemi_test_predictions_stemi_optimized_v3.npz
Dùng cột 'bag_ResNet1D' (cùng kiến trúc, cùng predictor bagging, from-scratch) để so sánh công bằng -- tách bạch hiệu ứng pretrain khỏi hiệu ứng ensemble 8 model.
{
  "point_delta_auprc": -0.00011088797075775592,
  "ci95_lo": -0.02794323442029658,
  "ci95_hi": 0.02910945170938825,
  "p_value_approx": 0.981,
  "significant": false
}

>>> Chênh lệch không có ý nghĩa thống kê -- không đủ bằng chứng để đổi.


In [13]:
def export_run_manifest():
    manifest = {
        "task": TASK,
        "run_mode": RUN_MODE,
        "created_at": datetime.utcnow().isoformat(),
        "source_backbone_manifest": str(_manifest_path),
        "source_backbone_sha256": _expected_sha256,
        "freeze_level_screening": {FREEZE_LEVELS[k]: v for k, v in screening_results.items()},
        "best_freeze_level": FREEZE_LEVELS[best_level],
        "threshold_policy": THRESHOLD_POLICY,
        "locked_threshold": LOCKED_THRESHOLD,
        "pool_oof_metrics": _oof_metrics,
        "test_metrics": test_metrics,
        "test_auprc": test_auprc,
        "test_auroc": test_auroc,
        "n_pool": len(pool_df), "n_test": len(test_df),
    }
    path = FINETUNE_OOF_DIR / f"{TASK}_finetune_run_manifest.json"
    path.write_text(json.dumps(manifest, indent=2, ensure_ascii=False))
    print(f"Manifest: {path}")
    return path

_ = export_run_manifest()

/tmp/ipykernel_626/1393943774.py:5: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "created_at": datetime.utcnow().isoformat(),


Manifest: /content/drive/MyDrive/ACS-ECG-AI_pretrain_finetune/manifests/stemi_finetune_oof/stemi_finetune_run_manifest.json


## Kết thúc

Kết quả (mức freeze/unfreeze thắng, threshold khoá, metric POOL-OOF/TEST, so sánh bootstrap với
baseline) đã lưu tại `manifests/stemi_finetune_oof/stemi_finetune_run_manifest.json`.

**Sẵn sàng chạy `RUN_MODE="full"`** ngay khi notebook 14 chạy xong (`RUN_MODE="full"`, đã sinh
`*_run_manifest.json` + `*_backbone_only.pt`) — Bước 2 đã đọc dữ liệu ACS-ECG 2026 thật và tái lập
đúng POOL/TEST/fold của `stemi_pipeline_optimized_v3.ipynb`.

**Việc còn lại chỉ có tính tham khảo, không chặn chạy full:**
- **Bước 10** — `baseline_y_prob`: cell đã tự tìm `stemi_test_predictions_*.npz` và liệt kê các
  cột model đơn có sẵn; điền 1 dòng để so nhanh, hoặc theo hướng dẫn ở markdown Bước 10 nếu muốn
  so đúng với ensemble champion của baseline.

Đưa kết quả cuối cùng vào `Bao_cao_nghien_cuu.docx` (mục 2.2 hoặc 2.3 tương ứng STEMI), diễn giải
theo đúng khung: đây là bước trong quy trình phát triển mô hình (giống "Fine-tune foundation model"
ở Phần I), không phải external validation.